# Point-treatment TMLE: did transition navigation improve the experience score?

This notebook estimates one average treatment effect from observational data with targeted
maximum likelihood estimation (TMLE). Each step shows its code, its output, and what the output
tells you. [Point-treatment TMLE](../technical-reference/point-treatment-tmle.md) gives the
parameter, the influence curve, and the algorithm.

## The applied question

A regional health plan offers adults a **standard transition-navigation protocol** when a discharge
home is ordered. The offer is a bedside plan and two scheduled contacts within 30 days. Nobody
randomized it, and discharge teams used a recorded risk process.

The program sponsor asks one question. How much would the mean 30-day transition score change if
every eligible discharge received the offer, rather than usual support? That question is the
average treatment effect (ATE), not a regression coefficient.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| say why an unadjusted difference is not a causal effect | Step 3 |
| write the study protocol before you fit anything | Step 4 |
| name the estimand and the assumptions that identify it | Step 5 |
| fit TMLE with explicit learners, and read its interval | Step 6 |
| tell the ATE, the ATT, and the ATC apart | Step 7 |
| see what double robustness does not promise | Step 8 |
| read the diagnostics and the truncation curve | Step 9 |
| read the sensitivity analysis | Step 10 |
| save a fit and replay its assessment | Step 11 |

## Why this method

| your situation | what this method buys | what it costs |
| --- | --- | --- |
| observational data, confounders measured | double robust point consistency: either consistent nuisance model can supply it, under positivity and regularity conditions | you must name the estimand first |
| the nuisance functions are not linear | flexible learners fit both nuisances, and the estimate stays a plug-in | a valid interval needs a product rate on the two nuisances, which Step 6 defines |
| you want an interval you can report | the interval comes from the targeted influence curve | positivity must hold, and a support report cannot verify it |

A regression coefficient and an inverse-probability-weighted mean each rest on one model. TMLE
targets the outcome regression with the treatment mechanism, so it uses both. The table below
defines the five terms this notebook uses most, and links each one to its reference. Later steps
link to the reference where another term first matters.

| term | plain meaning | reference |
| --- | --- | --- |
| estimand | the number the question asks for, written before any model is chosen | [estimands](../user-guide/estimands.md) |
| nuisance | a model the estimate needs but the question does not ask about. Here, the outcome regression Q and the treatment mechanism g | [point-treatment TMLE](../technical-reference/point-treatment-tmle.md) |
| targeting | a small update to Q, weighted by g, that removes the first-order bias of the plug-in estimate | [targeting and bounds](../user-guide/methods-learners.md#targeting-and-bounds) |
| influence curve | each row's contribution to the estimate's error. Its variance gives the standard error | [inference](../technical-reference/inference.md) |
| positivity | every kind of patient has some chance of each arm, so both arms have data to compare | [diagnostics](../user-guide/results-assessment.md#diagnostics) |

## Step 1: set up

The setup imports the learners and prints the installed `cleverly` version. Every fit below passes
its learners, fold count, and random seed explicitly, so a rerun reproduces the stored outputs.


In [1]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.2


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.


## Step 2: the data

The data come from a synthetic law with a known answer. `navigation_data` draws the rows and gives
the columns the program's names. The code prints the column names, the first rows, and the true
values of the law.

In [2]:
from cleverly.datasets import navigation_data

frame, truth = navigation_data(n=3_000, seed=21)
print("rows and columns:", frame.shape)
print("column names:", list(frame.columns))
print(frame.head().round(3))
print()
print("known values of the synthetic law:")
for key in ("ey1", "ey0", "ate", "att", "atc"):
    print(f"  {key}: {truth[key]:.3f}")

rows and columns: (3000, 6)
column names: ['transition_score', 'transition_navigation', 'discharge_risk', 'prior_utilization', 'medication_burden', 'age']
   transition_score  transition_navigation  discharge_risk  prior_utilization  medication_burden    age
0             0.748                    0.0           0.359              1.511             -1.786  1.687
1             0.554                    1.0          -0.047             -0.800             -0.803 -1.083
2             0.463                    0.0          -0.224              0.834              0.584  0.638
3             0.084                    0.0          -1.695             -1.571              1.554  0.969
4             0.948                    1.0           2.183              1.210             -1.024  1.285

known values of the synthetic law:
  ey1: 0.567
  ey0: 0.404
  ate: 0.163
  att: 0.179
  atc: 0.149


**What this output tells you.** Each row is one discharge. The helper `navigation_data` draws the
rows from the generator `make_nonlinear_bounded`. `transition_navigation` is 1 for an offer and 0
for usual support. The four baseline covariates are standardized (mean 0, SD 1), so a negative
`age` is below the average age. The transition score is a share of the maximum score, so every
value lies between 0 and 1.

That support belongs to the measurement instrument, not to the draw. Step 6 declares it to the
fit.

The true ATE is 0.163. The effect is larger than a real navigation program would expect, so each
fit shows its behavior clearly. Every discharge is independent here, and the
[cross-fitting tutorial](cross-fitting.ipynb) adds shared navigator teams.

| feature of the law | what it means in this program |
| --- | --- |
| the four baseline covariates drive assignment and the outcome | higher-risk patients are more likely to receive an offer and report different outcomes. All four are confounders in the synthetic law |
| both nuisance functions are nonlinear | a GLM is misspecified for each one, which is the condition this page exploits |
| the score has a known support | every value lies in the open interval from 0 to 1, so a fit can declare that support rather than read a scale from the data |
| the effect varies with the covariates | `ate`, `att`, and `atc` differ (0.163, 0.179, and 0.149), so the estimand must be named rather than inferred |

A real program has no `truth`. Every comparison against it below is a teaching device.


## Step 3: association first

A confounder is a variable that changes both who receives the offer and the outcome. The code
compares the two arms before any adjustment. It prints the mean score and the mean of each
baseline covariate by arm.


In [3]:
covariates = ["discharge_risk", "prior_utilization", "medication_burden", "age"]
by_arm = frame.groupby("transition_navigation")[["transition_score", *covariates]].mean()
print(by_arm.round(3))
print()
print("share offered navigation:", round(float(frame["transition_navigation"].mean()), 3))
unadjusted = by_arm.loc[1.0, "transition_score"] - by_arm.loc[0.0, "transition_score"]
print(f"unadjusted difference in mean score: {unadjusted:.3f}")
print(f"population ATE:                      {truth['ate']:.3f}")
print(f"population ATT:                      {truth['att']:.3f}")

                       transition_score  discharge_risk  prior_utilization  medication_burden    age
transition_navigation                                                                               
0.0                               0.386          -0.241              0.005             -0.026 -0.018
1.0                               0.589           0.259              0.010             -0.045  0.062

share offered navigation: 0.458
unadjusted difference in mean score: 0.203
population ATE:                      0.163
population ATT:                      0.179


**What this output tells you.** The offered patients score 0.203 higher on average. The
true ATE is 0.163, and the true ATT is 0.179. The arms differ before the offer. The mean
`discharge_risk` is 0.259 among offered patients and -0.241 among the others.

The unadjusted difference compares two groups with different covariates, so it does not answer
the ATE question. In this law, a higher `discharge_risk` raises both the chance of an offer and
the size of the effect. The offered group therefore gains more than the average patient, and the
ATT exceeds the ATE. The offered patients would also score higher than the others under usual
support alone. Both distortions raise the unadjusted difference above the ATE here, so its size
measures neither one.

On this draw, the unadjusted difference is closer to the ATT than to the ATE. An estimator must
compare like with like. The next two steps state which comparison the question needs, and which
assumptions make it possible.


## Step 4: write the protocol

A `StudyProtocol` records the scientific design before any model runs. It follows target-trial
vocabulary. You name the population, the time zero, the strategies, the outcome, and how later
events are handled. The record gets a fingerprint, and every result fitted from it carries that
fingerprint.

`navigation_protocol()` in `cleverly.datasets` holds the program's protocol, so every tutorial
starts from one record. A real analysis calls `StudyProtocol(...)` with the same ten fields. The
code prints each field.

In [4]:
from cleverly.datasets import navigation_protocol

protocol = navigation_protocol()
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; 623fc2c240615d26
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and before the navigation offer
treatment strategies: ['Offer standard transition navigation', 'Provide usual discharge support']
treatment versions: ['Bedside transition plan and two scheduled navigator contacts within 30 days', 'No access to the transition-navigation offer']
outcome: Patient-reported transition score, as a share of the maximum score
horizon: 30 days after discharge
intercurrent-event handling: ['Use the transition score regardless of readmission', 'Analyze the offer regardless of completed contacts', 'The protocol scores death before day 30 as the worst transition score (composite strategy)']
interference unit: Individual patient
assumption rationale: ['The r

**What this output tells you.** The first line gives the schema version and the fingerprint
`623fc2c240615d26`. The other lines repeat each field. Read them as a checklist.

| protocol field | the question it answers for this program |
| --- | --- |
| target population and eligibility | who the effect is about |
| time zero | when follow-up starts. Here, the discharge-home order, before the offer |
| treatment strategies and versions | what "offer" and "usual support" mean in practice |
| outcome and horizon | what is measured, on what scale, and when |
| intercurrent-event handling | what happens to readmission, incomplete contacts, and death before day 30 |
| interference unit | whose assignment can affect whose outcome |
| assumption rationale | why the design supports the identification assumptions |

The outcome field names the scale as a share of the maximum score. That sentence is the evidence
for the support Step 6 declares. A fit cannot recover it from the data.

Time zero and eligibility coincide at the discharge-home order, before the offer. Eligibility
therefore cannot depend on the offer. The protocol does not store the contrast. The typed estimand in the
next step owns it.


## Step 5: design and identification

The design says which column plays which role. The estimand says which contrast you want. The two
are separate on purpose, so a later change of estimator cannot change the question.

Identification turns a causal question into a quantity the data can estimate. The `identify` call
returns that formula, the nuisance models it needs, and the assumptions that make it causal.


In [5]:
from cleverly import ATE, CausalStudy, PointTreatment

study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=tuple(covariates),
    ),
    protocol=protocol,
)
effect = study.identify(ATE(reference=0))

print(effect.summary())

average treatment effect, E[Y^a] - E[Y^reference]
identified by explicit-adjustment: E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
adjustment/history: ['discharge_risk', 'prior_utilization', 'medication_burden', 'age']
required nuisances: ['outcome_regression', 'treatment_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 623fc2c240615d26
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measu

**What this output tells you.** The first lines show the estimand and its observed-data formula.
The formula averages the outcome regression over all patients, once with the offer and once
without. The required nuisances are the outcome regression Q and the treatment mechanism g, the
probability of an offer given the covariates. The summary then lists four assumptions and repeats
the stored protocol.

| assumption | what it means for this program | can the data check it? |
| --- | --- | --- |
| consistency | an offer always means the declared bedside plan and two scheduled contacts | no |
| no interference | one patient's assignment does not change another patient's offer or outcome | no |
| no unmeasured confounding | the recorded baseline variables block every common cause of assignment and the score | no |
| positivity | each baseline profile has some chance of an offer and of usual support | partly, through the support report |

The [shared study design](index.md#the-shared-study-design) states how the program supports
consistency and no interference. This page changes nothing in it.

No unmeasured confounding needs a causal argument. For example, an unrecorded discharge-team
judgment that affects both assignment and recovery would violate it. No estimator repairs that
failure. The sensitivity step asks how strong such a judgment would need to be.

The synthetic law needs only four covariates. A real protocol should also evaluate pre-assignment
site, navigator-team, calendar, and language-access causes. Add them when the causal review places
them on a common-cause path.


## Step 6: estimate the ATE

The configuration is written out in full, so you see every choice.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | gradient boosting | fits Q, the expected score given the offer and the covariates |
| `treatment_learner` | gradient boosting | fits g, the probability of an offer given the covariates |
| `CrossFitting(n_folds=5)` | five folds | predicts each row from models that did not see that row |
| `Targeting(q_bounds=(0.0, 1.0))` | the declared score support | fixes the outcome scale before any fold is drawn |
| `Inference(alpha=0.05)` | 95% interval | sets the interval level |
| `Runtime(random_state=21, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |

Cross-fitting splits the rows into folds. Each nuisance prediction for a row comes from models fit
on the other folds. The [methods guide](../user-guide/methods-learners.md#two-fold-layers)
explains the two fold layers.

A cross-fitted fit of a continuous outcome must declare `q_bounds`. Step 4 recorded the scale, so
this fit states it. With `q_bounds=None` the fit would read the scale from every observed score,
including the rows it holds out, and `cleverly` refuses that fit.

Targeting then runs once on the pooled out-of-fold predictions. It updates Q along the clever
covariate until the estimated efficient score equation is solved. The clever covariate is one over
g for the arm a row received, with a negative sign for usual support. Rows with a rare assignment
therefore move the update most.
[Targeting and bounds](../user-guide/methods-learners.md#targeting-and-bounds) gives the options.


In [6]:
from cleverly import CrossFitting, Inference, ModelSpec, Runtime, Targeting, TMLEMethod

flexible = TMLEMethod(
    models=ModelSpec(
        outcome_learner=HistGradientBoostingRegressor(random_state=21),
        treatment_learner=HistGradientBoostingClassifier(random_state=21),
    ),
    cross_fitting=CrossFitting(n_folds=5),
    targeting=Targeting(q_bounds=(0.0, 1.0)),
    inference=Inference(alpha=0.05),
    runtime=Runtime(random_state=21, n_jobs=1),
)
result = effect.estimate(method=flexible)

estimate = result["ate"]
print(result.summary())
print()
print(f"estimate:        {estimate.psi:.3f}")
print(f"standard error:  {estimate.std_error:.3f}")
print(f"95% CI:          ({estimate.ci[0]:.3f}, {estimate.ci[1]:.3f})")
print(f"population ATE:  {truth['ate']:.3f}")

Targeted maximum likelihood estimation
n = 3000; covariates = 4; P(A=1) = 0.4583
causal estimand: average treatment effect, E[Y^a] - E[Y^reference]
identification: explicit-adjustment; E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
required nuisances: outcome_regression, treatment_mechanism
identification assumptions: consistency: Y = Y^a when A = a; no interference: one unit's potential outcome does not depend on other units' treatment assignments; no unmeasured confounding: Y^a is independent of A given W; positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 623fc2c240615d26
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseli

**What this output tells you.** The summary repeats the estimand, the identification, and the
protocol with its fingerprint. It then reports the method configuration. It names the construction
as stacked CV-TMLE, with nuisances cross-fitted over 5 folds. It also shows the propensity
truncation bound, [0.0114, 0.9886].

The summary prints no `outcome scaled` line. The declared support already matches the score's own
scale, so the fit rescales nothing. The line `fluctuation: logistic (iterative)` says that the
update runs on the logistic scale of that score. The targeted predictions therefore stay inside
the declared bounds.

The estimate is 0.172 with a standard error of 0.010. The 95% interval is (0.153, 0.191), and it
contains the true ATE of 0.163. That is one draw, not a coverage result.

The interval is built from the targeted influence curve, not from the outcome model's own
standard error. Its validity remains conditional on support, nuisance convergence, the product-rate
condition, and the declared dependence structure. The product-rate condition asks that the errors
of Q and g, multiplied together, shrink faster than one over the square root of n. The
[cross-fitting tutorial](cross-fitting.ipynb) shows why the fold separation matters for flexible
learners.


## Step 7: which population is the number about?

The average treatment effect answers a question about every patient. A spread decision asks
something narrower.

| estimand | the question it answers | who asks it |
| --- | --- | --- |
| ATT | what did patients who received an offer gain from assignment? | the teams reviewing the rollout |
| ATE | what would the eligible population gain if everyone received an offer? | the program sponsor |
| ATC | what would patients who received usual support gain from an offer? | whoever is deciding on spread |

These are three parameters, not three estimates of one. A second law makes the gap visible, because
its effect modification is aligned with the propensity. Patients most likely to receive an offer
benefit most. The code fits all three with linear learners and prints each beside its true value.

The second law has a Gaussian outcome with no known finite support, so a cross-fitted fit could
declare no `q_bounds`. These three fits therefore run in sample with `CrossFitting(enabled=False)`,
which draws no split. The question here is which population the number describes.


In [7]:
from cleverly import ATC, ATT
from cleverly.datasets import make_heterogeneous

spread_frame, spread_truth = make_heterogeneous(n=3_000, seed=23)
spread_frame = spread_frame.rename(
    columns={
        "Y": "transition_score",
        "A": "transition_navigation",
        "W1": "discharge_risk",
        "W2": "medication_burden",
    }
)
spread_study = CausalStudy(
    spread_frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=("discharge_risk", "medication_burden"),
    ),
)
simple = TMLEMethod(
    models=ModelSpec(
        outcome_learner=LinearRegression(n_jobs=1),
        treatment_learner=LogisticRegression(max_iter=1000, random_state=23),
    ),
    cross_fitting=CrossFitting(enabled=False),
    runtime=Runtime(random_state=23, n_jobs=1),
)
spread_points = {}
for estimand, key in (
    (ATT(reference=0), "att"),
    (ATE(reference=0), "ate"),
    (ATC(reference=0), "atc"),
):
    point = spread_study.identify(estimand).estimate(method=simple)[key]
    spread_points[key] = point
    low, high = point.ci
    print(
        f"{key}: {point.psi:6.3f}  CI=({low:.3f}, {high:.3f})  population {spread_truth[key]:.3f}"
    )

att:  1.724  CI=(1.567, 1.881)  population 1.691
ate:  1.008  CI=(0.893, 1.123)  population 1.000
atc:  0.317  CI=(0.158, 0.476)  population 0.309


**What this output tells you.** The three estimates are 1.724, 1.008, and 0.317. Their true values
are 1.691, 1.000, and 0.309. The population law has `att > ate > atc` by construction.

Its propensity is exactly logistic, so the logistic treatment learner is correctly specified. The
linear outcome learner omits the law's navigation-by-risk interaction, so these fits rest on the
treatment model. Do not use interval overlap as a test of the differences between these
parameters.

Read the three values as a warning about spread. The offered patients' gain is the ATT. Patients
who received usual support would get the ATC, which here is a small fraction of it. A program that
budgets the network rollout against the ATT will overpromise.


## Step 8: the failure mode, both nuisance models misspecified

Double robustness means the point estimate stays consistent when *either* Q *or* g is consistent.
It is a claim about *or*, not about *and*. Use the known synthetic law to compare learner
combinations. Treat the result as an illustration, not as validation evidence.

A gradient-boosted learner can represent the law's nonlinear features. The linear learners omit
those features by construction. Three more fits show the resulting finite-sample pattern. Each fit
copies the Step 6 method and changes only the learners. Each line prints the estimate, its interval,
and its distance from the true ATE.

In [8]:
from dataclasses import replace

dr_points = {}


def fit(outcome_learner, treatment_learner, label):
    method = replace(
        flexible,
        models=ModelSpec(outcome_learner=outcome_learner, treatment_learner=treatment_learner),
    )
    point = effect.estimate(method=method)["ate"]
    dr_points[label] = point
    low, high = point.ci
    miss = abs(point.psi - truth["ate"])
    print(f"{label:22s} psi={point.psi:6.3f}  CI=({low:.3f}, {high:.3f})  miss={miss:.3f}")


fit(
    HistGradientBoostingRegressor(random_state=21),
    LogisticRegression(max_iter=1000, random_state=21),
    "flexible Q, linear g",
)
fit(
    LinearRegression(n_jobs=1),
    HistGradientBoostingClassifier(random_state=21),
    "linear Q, flexible g",
)
fit(
    LinearRegression(n_jobs=1),
    LogisticRegression(max_iter=1000, random_state=21),
    "both linear",
)
print(f"population ATE: {truth['ate']:.3f}")

flexible Q, linear g   psi= 0.170  CI=(0.160, 0.180)  miss=0.007


linear Q, flexible g   psi= 0.171  CI=(0.142, 0.200)  miss=0.008
both linear            psi= 0.141  CI=(0.127, 0.155)  miss=0.021
population ATE: 0.163


**What this output tells you.** With one flexible learner, the fits miss the true ATE by 0.007 and
0.008. Each of those intervals contains the true value of 0.163. With both learners linear, the
fit misses by 0.021, and its interval (0.127, 0.155) excludes that value. On this draw, the fit with
both linear learners misses the population value by about three times either fit with one flexible
learner.

This output does not establish nuisance consistency or repeated-sampling coverage.

| caution | why |
| --- | --- |
| the linear models omit known terms | a real analysis does not reveal which nuisance model is consistent |
| one draw is not a coverage result | coverage is a repeated-sampling property |
| one nuisance is inconsistent | the point estimate can stay consistent, but the influence-curve interval need not be valid. [DR-TMLE](dr-tmle.ipynb) addresses that case |
| point consistency and interval validity differ | Wald inference needs the stated product-rate and regularity conditions |


## Step 9: diagnostics, what the fit can check

Start with the combined assessment of the Step 6 fit. It presents validation, diagnostics, and
sensitivity together. `assessment.attention` lists the failure and warning rows. The code then
prints three retained reports: support, nuisance models, and score equations.

The support report describes overlap in the fitted data. Overlap is the sample view of positivity,
which the terms table defines. The report shows how the fitted propensities spread, how many rows
hit the truncation bound, and how concentrated the weights are.

In [9]:
assessment = result.assess()
print(assessment.summary())
print("needs attention:", tuple(item.name for item in assessment.attention))

support = assessment.report("support")
nuisance = assessment.report("nuisance_models")
scores = assessment.report("score_equations")
print()
print(support.summary())
print()
print(nuisance.summary())
print()
print(scores.summary())

Returned results
----------------
surface      operation  result                                                                                                                                                                                   
-----------  ---------  -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
validation   support    maximum truncated fraction 0.7%; minimum effective-sample-size ratio 26.9%; group load: mean:h1 370.2/3000 Kish-equivalent mask rows (12.3%; 12.3% all; draw 01 of 01); not estimator ESS
sensitivity  evalue     point=3.31, limit=3.019, source scale=mean difference; approximate conversion                                                                                                            

Checks
------
status   count  operations                
-------  -----  --------------------------
warning  1      validatio

**What this output tells you.** Read the four parts in order.

| output part | what it shows on this draw |
| --- | --- |
| `Checks` and `needs attention` | one warning, `nuisance_models`. The score-equation check passed |
| positivity and overlap | 21 units (0.70%) sit at the truncation bound. The treated arm keeps an effective-sample-size ratio of 0.269 |
| nuisance model diagnostics | the boosted propensity is poorly calibrated, with a calibration slope of 0.46 against an ideal of 1 |
| score-equation check | targeting solved the estimated efficient score equation |

The `Returned results` rows are calculations that ran. They are not passes. The
omitted-variable rows are `unavailable` here, and Step 10 reads the refusal. The call
`assessment.to_frame()` gives the complete row ledger. On a first reading,
skip the table with the `Kish-equivalent rows` column. It describes weight concentration and
applies no threshold.

Two lessons follow. First, a calibration slope below 1 means the fitted propensities are more
extreme than the observed offer rates. Extreme propensities give large weights, and the treated
arm's ratio of 0.269 shows that concentration.
[Collaborative TMLE](collaborative-tmle.ipynb) chooses the assignment model by its effect on the
targeted estimate rather than by prediction alone.

Second, the report gives no positivity verdict, because no universal cutoff applies. The support
report describes fitted overlap and cannot verify population positivity.


### The truncation curve

Truncation clips each fitted propensity into a bound, so no row gets an extreme weight. The code
calls `result.diagnostics.truncation_curve()`. It retargets the estimate at a default grid of
bounds and at the fitted bound, without a refit of the nuisance models.

In [10]:
curve = result.diagnostics.truncation_curve()
columns = ["bound", "psi", "std_err", "ci_lower", "ci_upper", "truncated_fraction"]
print(curve[columns].round(4).to_string(index=False))

 bound    psi  std_err  ci_lower  ci_upper  truncated_fraction
0.0010 0.1721   0.0099    0.1527    0.1914              0.0000
0.0050 0.1718   0.0099    0.1525    0.1911              0.0017
0.0100 0.1717   0.0099    0.1524    0.1910              0.0057
0.0114 0.1718   0.0098    0.1526    0.1910              0.0070
0.0250 0.1729   0.0089    0.1555    0.1902              0.0223
0.0500 0.1730   0.0080    0.1574    0.1887              0.0490
0.1000 0.1721   0.0070    0.1584    0.1859              0.1207
0.2000 0.1716   0.0059    0.1600    0.1833              0.2800


**What this output tells you.** Each row is one bound. The fitted bound is 0.0114, where the
estimate is 0.1718, the Step 6 estimate. Across bounds from 0.0010 to 0.2000, the estimate stays
between 0.1716 and 0.1730. The standard error falls from 0.0099 at the smallest bound to 0.0059 at
the largest. From the fitted bound onward it falls at every step.

The largest bound clips a fraction of 0.2800 of the rows, so it changes the weight of more than a
quarter of the patients. A larger bound lowers the variance and can add truncation bias that the
standard error does not include. A narrower interval at a larger bound is therefore not a precision gain.

The fitted bound 0.0114 comes from the default rule $5 / (\sqrt{n} \log n)$ at $n = 3000$. The
movement shows sensitivity to this regularization choice. Limited movement does not verify
positivity.


## Step 10: sensitivity, what the fit cannot check

Diagnostics cannot see unmeasured confounding.
[Sensitivity analysis](../user-guide/results-assessment.md#sensitivity-analysis) asks how strong a
confounder would need to be to explain the result away. A benchmark calibrates that strength
against a covariate you did measure.

| term | plain meaning |
| --- | --- |
| `cf_y` | the share of the remaining outcome variation that a hidden confounder explains |
| `cf_d` | the share of the remaining treatment variation that a hidden confounder explains |
| `rho` | how closely the confounder's two effects align. `rho=1` is the worst case |
| robustness value | the equal `cf_y` and `cf_d` that moves the point estimate to zero at `rho=1` |
| $\nu^2$ | the second moment of the Riesz representer of the estimand. The largest bias is $\sqrt{\sigma^2 \nu^2}$, and every number below scales with it |

Each bound needs an estimate of $\nu^2$, and `cleverly` implements two. The default,
`nu2_estimator="auto"`, resolves to the doubly robust form. The code first calls the robustness
value with that default, which refuses here. It then passes `nu2_estimator="plugin"` to each call,
and the reading states what that choice rests on.

Each benchmark refits the nuisances once without that covariate. The call goes to the facade
because `assess(include_refits=True)` would also run the costlier refutations. The bounds call
passes `rho=1.0` explicitly, so the bounds are worst-case.

In [11]:
from cleverly import CapabilityError

try:
    result.sensitivity.robustness_value()
except CapabilityError as refusal:
    nu2_refusal = str(refusal)
    print("the default estimator refused:", nu2_refusal)
else:
    raise AssertionError("the doubly robust nu^2 was positive on this fit")
print()

robustness = result.sensitivity.robustness_value(nu2_estimator="plugin")
strong = result.sensitivity.benchmark(covariates=("discharge_risk",), nu2_estimator="plugin")
benchmark = result.sensitivity.benchmark(covariates=("medication_burden",), nu2_estimator="plugin")
bounds = result.sensitivity.omitted_confounding(
    cf_y=benchmark.cf_y, cf_d=benchmark.cf_d, rho=1.0, nu2_estimator="plugin"
)
print(f"robustness value: {robustness['rv']:.3f}")
assert "rva" not in robustness
print(strong)
print()
print(benchmark)
print(f"bias-adjusted bounds at the benchmark strength: ({bounds.lower:.3f}, {bounds.upper:.3f})")
try:
    lower_limit = bounds.ci_lower
except CapabilityError as limit_error:
    limit_refusal = str(limit_error)
    print("the one-sided limits refused:", limit_refusal)
else:
    raise AssertionError(f"the plug-in bound reported a confidence limit, {lower_limit}")

the default estimator refused: the 'doubly_robust' estimator of nu^2 returned -7.96655 for 'ate', and a second moment cannot be negative. E[2 m(alpha_hat) - alpha_hat^2] equals nu_0^2 minus the squared error of the fitted representer, so this value reports a treatment mechanism the bound's derivation does not cover. The plug-in E[alpha_hat^2] squares that same fitted representer, so it is not a substitute. (nu2_estimator='auto' resolved to 'doubly_robust'.)



robustness value: 0.233
Benchmark for 'ate' against ['discharge_risk']
------------------------------------------------
quantity  with covariates  without 
--------  ---------------  --------
estimate  0.17181          0.23578 
sigma^2   0.01748          0.040486
nu^2      23.762           21.071  

implied cf_y = 1.0000, cf_d = 0.1277, rho = 0.2571
the estimate moved by +0.063964 when these covariates were dropped

Benchmark for 'ate' against ['medication_burden']
------------------------------------------------
quantity  with covariates  without 
--------  ---------------  --------
estimate  0.17181          0.17198 
sigma^2   0.01748          0.019575
nu^2      23.762           16.642  

implied cf_y = 0.1198, cf_d = 0.4278, rho = 0.0014
the estimate moved by +0.00017035 when these covariates were dropped
bias-adjusted bounds at the benchmark strength: (-0.021, 0.365)
the one-sided limits refused: SensitivityBounds.ci_lower is not defined under nu2_estimator='plugin': no derivation 

**What this output tells you.** The first call refuses. The doubly robust estimator of $\nu^2$
returned -7.96655 for `ate`, and a second moment cannot be negative. That estimator equals the true
$\nu_0^2$ minus the squared error of the fitted representer. A negative value therefore reports a
fitted treatment mechanism far from the treatment law. Step 9 reports the same problem from the
other side: the propensity calibration slope is 0.4615 against an ideal of 1.

The code then asks for the plug-in estimator by name. That estimator squares the same fitted
representer, so it repairs no mechanism. It only drops the correction term that exposed the
distance. Every number below therefore rests on the fitted propensity, and this fit gives you a
reason to doubt it. Read the numbers as a worked example of the vocabulary. Do not report them as a
bound for this fit.

The robustness value is 0.233. At `rho=1`, equal strengths of 0.233 move the point estimate to
zero.

The first benchmark drops `discharge_risk`, the covariate behind the recorded risk process. The
estimate moves from 0.17181 to 0.23578. The implied `cf_y` reaches 1.0000, which is the whole of
the remaining outcome variation. A hidden confounder as strong as the recorded risk score would
leave the outcome model nothing else to explain. The bound formula needs a `cf_y` below that
ceiling, so this covariate calibrates no bound here.

The second benchmark drops `medication_burden`. Its implied strengths are `cf_y = 0.1198` and
`cf_d = 0.4278`. This covariate explains little of the remaining outcome variation and much of the
remaining treatment variation. Its role is the mirror image of the first covariate's. The bias
grows with the product of the two strengths, not with either one alone.

The code keeps `rho=1` rather than the implied `rho = 0.0014`, so the bounds stay worst-case. At
the benchmark strengths, the bias-adjusted bounds are (-0.021, 0.365). The range contains zero. A
confounder as strong as `medication_burden`, at the worst alignment, would be enough to explain
this estimate away.

The review must decide whether an unrecorded team judgment could be that strong. The
[omitted-variable bounds](../technical-reference/validation-methods.md#omitted-variable-bounds-robustness-value-benchmark-and-contours)
section defines each quantity. `result.assess(include_refits=True)` adds placebo, noise, and
subsampling refutations. A stable refutation is not evidence of correctness.

Reading `ci_lower` refuses. The plug-in $\nu^2$ moves with the fitted propensity at first order.
No derivation in a source this package cites gives the standard error of a bound built on it. The
package therefore reports no one-sided limit and no confidence-limit value here. The doubly robust
estimator reports both, and it refused on this fit.

## Step 11: keep the result

A fit is an artifact. It carries the nuisance models, the influence curves, and the provenance
stamp, so the assessment replays without refitting. A program that reports quarterly needs that.
The code saves the result to a temporary directory, loads it, and checks the protocol fingerprint.


In [12]:
from pathlib import Path
from tempfile import TemporaryDirectory

from cleverly import load

with TemporaryDirectory() as directory:
    saved = Path(directory) / "transition-navigation-ate.joblib"
    result.save(saved)
    restored = load(saved)
    restored_protocol = restored.identified_effect.protocol
    assert restored_protocol is not None
    assert restored_protocol.fingerprint == restored.provenance.protocol_fingerprint
    print("restored protocol fingerprint:", restored_protocol.fingerprint)
    print(restored.replayability)
    replayed = restored.assess()
    print("needs attention after the restore:", tuple(item.name for item in replayed.attention))

restored protocol fingerprint: 623fc2c240615d26
Replayability(summarize_existing_artifacts=True, retarget_cached_nuisances=True, evaluate_stored_representer=False, refit_nuisances=True, evaluate_new_data=False, unreconstructible=())
needs attention after the restore: ('nuisance_models',)


**What this output tells you.** The restored protocol has the fingerprint `623fc2c240615d26`.
Step 4 printed the same fingerprint. The replayed assessment names the same warning as Step 9. The
`Replayability` line says which operations the restored artifact can still perform.

| flag | value | meaning |
| --- | --- | --- |
| `summarize_existing_artifacts` | True | the stored diagnostics can be summarized again |
| `retarget_cached_nuisances` | True | targeting can run again without a nuisance refit, as the truncation curve does |
| `evaluate_stored_representer` | False | the artifact cannot evaluate a different parameter from its stored fit |
| `refit_nuisances` | True | the artifact keeps the method configuration, so it can refit the nuisance models |
| `evaluate_new_data` | False | the artifact cannot score new discharges |

Load only joblib files you trust, and keep the dependency versions compatible.


## How far to trust this

The [stacked point-treatment CV-TMLE study](../technical-reference/method-evidence/stacked-point-treatment-cv-tmle.md)
validates this construction with GLM learners, ten folds, and propensity bounds of 0.025 to 0.975.
It adds a cross-fitted tree-learner control. No registered study covers this fit's boosted
learners, five folds, or automatic bound of 0.0114.

| layer | establishes | does not establish |
| --- | --- | --- |
| assessment overview | which stored checks need attention, and which operations did not run | the detail needed to interpret each retained report |
| retained diagnostics | that targeting converged, and how concentrated the fitted weights are | that the nuisance models are right |
| sensitivity analysis | how strong an unmeasured confounder would need to be, if the fitted propensity is right | that no such confounder exists, or that the plug-in $\nu^2$ of Step 10 belongs to the treatment law |
| the registered study | the implementation recovers known truths and behaves as its theory predicts | that your identification assumptions hold on your data |

Nothing in this list validates the causal reading. That rests on consistency, no interference, and
no unmeasured confounding. All three are arguments about the program rather than about the fit.

## Where to go next

This page treated discharges as independent rows. Read
[CV-TMLE and cross-fitting](cross-fitting.ipynb) for the same question at network scale, where patients
share navigator teams. If your worry is instead which baseline variables belong in the assignment
model, read [collaborative TMLE](collaborative-tmle.ipynb).

The [examples index](index.md#the-program) lists every tutorial in the program.
